In [1]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv("required csv/diabetes_binary_health_indicators_BRFSS2015.csv")

X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Base Model
base_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

start = time.time()
base_model.fit(X_train, y_train)
base_time = time.time() - start

# Accuracies
base_train_acc = accuracy_score(y_train, base_model.predict(X_train))
base_test_acc = accuracy_score(y_test, base_model.predict(X_test))

print("Base Model Results")
print("Train Acc:", base_train_acc)
print("Test Acc:", base_test_acc)
print("Training Time:", base_time)


c:\Users\vimal\anaconda3\envs\gpu_lab\lib\site-packages\xgboost\training.py:199: UserWarning: [15:23:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Base Model Results
Train Acc: 0.8615184484389783
Test Acc: 0.850461210974456
Training Time: 0.7103710174560547


# RANDOM SAMPLING (Row Subsampling Scaling)

In [3]:
# Random Sampling (50% of training data)
sample_frac = 0.75

X_train_sampled = X_train.sample(frac=sample_frac, random_state=42)
y_train_sampled = y_train.loc[X_train_sampled.index]

# Train sampled model
sampled_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

start = time.time()
sampled_model.fit(X_train_sampled, y_train_sampled)
sampled_time = time.time() - start

# Accuracies
sampled_train_acc = accuracy_score(y_train_sampled, sampled_model.predict(X_train_sampled))
sampled_test_acc = accuracy_score(y_test, sampled_model.predict(X_test))

print("\nRandom Sampling Results")
print("Train Acc:", sampled_train_acc)
print("Test Acc:", sampled_test_acc)
print("Training Time:", sampled_time)


c:\Users\vimal\anaconda3\envs\gpu_lab\lib\site-packages\xgboost\training.py:199: UserWarning: [15:24:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Random Sampling Results
Train Acc: 0.8651187848207715
Test Acc: 0.8496333964049196
Training Time: 0.31327033042907715


Random sampling was used to reduce the effective dataset size from 
𝑛
n to 
𝑘
k, where 
𝑘
<
𝑛
k<n, enabling faster model training. Since XGBoost has approximately 
𝑂
(
𝑛
⋅
𝑑
⋅
log
⁡
𝑛
)
O(n⋅d⋅logn) time complexity, reducing the number of samples directly decreases computational cost. Stratified sampling preserves class distribution while allowing empirical comparison between scaled and full-data models.

---

# JL Transform

In [4]:
from sklearn.random_projection import GaussianRandomProjection

# JL Transform (reduce features from 21 → 10)
jl = GaussianRandomProjection(n_components=10, random_state=42)

X_train_jl = jl.fit_transform(X_train)
X_test_jl = jl.transform(X_test)

# Train JL-transformed model
jl_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

start = time.time()
jl_model.fit(X_train_jl, y_train)
jl_time = time.time() - start

# Accuracies
jl_train_acc = accuracy_score(y_train, jl_model.predict(X_train_jl))
jl_test_acc = accuracy_score(y_test, jl_model.predict(X_test_jl))

print("\nJL Transform Results")
print("Train Acc:", jl_train_acc)
print("Test Acc:", jl_test_acc)
print("Training Time:", jl_time)


c:\Users\vimal\anaconda3\envs\gpu_lab\lib\site-packages\xgboost\training.py:199: UserWarning: [15:25:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



JL Transform Results
Train Acc: 0.8550585383159887
Test Acc: 0.8445482497634815
Training Time: 0.38402295112609863
